<a href="https://colab.research.google.com/github/karan124381/unmasked/blob/main/training/Unmaskd_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unmaskd — Colab Training Pipeline

This notebook prepares the training environment, downloads the Kaggle video dataset after you authenticate, inspects its actual layout, and prepares the first video training run. It does **not** commit datasets or credentials to GitHub.

Audio training is included as a documented ASVspoof 2021 DF setup because that dataset requires registration/download and should not be silently scraped.


## 0. Runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** if available.


In [1]:
!nvidia-smi || true
!git clone -q https://github.com/karan124381/unmasked.git /content/unmasked
%cd /content/unmasked
!pip install -q -r training/requirements.txt kaggle


Wed Oct  7 23:37:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Kaggle authentication

On your computer, download `kaggle.json` from Kaggle Account → API → Create New Token. Then use the Colab file-upload cell below. Do **not** paste the token into code or commit it.


In [4]:
from google.colab import files
uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Upload the Kaggle API token named kaggle.json'
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!rm -f kaggle.json
print('Kaggle credentials installed for this Colab runtime.')


Saving kaggle.json to kaggle.json
Kaggle credentials installed for this Colab runtime.


## 2. Download the video dataset

This uses Kaggle's official competition CLI. You must accept the competition rules on Kaggle if prompted. The download can be large, so make sure the Colab runtime has enough storage.


In [6]:
!mkdir -p /content/unmaskd_data/video_raw
!kaggle competitions files deepfake-detection | head -30
!kaggle competitions download deepfake-detection -p /content/unmaskd_data/video_raw --unzip


Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U7obCYAi4TPCYcWyMpdE-IugLSQwBcg50SfOyg0Gs1kC3VplB6KIkoSxGAX6R2DYz7g-O4XlzRuYxNJocXup03EKIPiluTx4qvca-hys1-pP6GvHMRViPvkRVGIfrBIRigcDESQyUkzleVH73ZYcqMhx9fvNa-j
name                                                 size  creationDate                
---------------------------------------------  ----------  --------------------------  
deep-fake-dataset/Celeb-Youtube-fake/0.mp4        2203086  2021-12-10 13:17:39.099000  
deep-fake-dataset/Celeb-Youtube-fake/1.mp4        1538579  2021-12-10 13:17:39.110000  
deep-fake-dataset/Celeb-Youtube-fake/10.mp4       2139221  2021-12-10 13:17:39.177000  
deep-fake-dataset/Celeb-Youtube-fake/100.mp4      3526883  2021-12-10 13:17:39.210000  
deep-fake-dataset/Celeb-Youtube-fake/1000.mp4     1009221  2021-12-10 13:17:39.202000  
deep-fake-dataset/Celeb-Youtube-fake/1001.mp4     1262289  2021-12-10 13:17:39.191000  
deep-fake-dataset/Celeb-Youtube-fake/1002.mp4      881397  2021-12-10 13:17:39.264000  
deep-

## 3. Inspect the actual Kaggle layout

Do this before assigning labels. Different deepfake datasets use different folder/file conventions; we should never guess that a filename means REAL or FAKE.


In [8]:
!kaggle competitions download deepfake-detection -p /content/unmaskd_data/video_raw

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


In [9]:
!pip install -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


In [10]:
!kaggle --version


Kaggle CLI 2.2.4


In [11]:
!kaggle --version

!rm -rf /content/unmaskd_data/video_raw
!mkdir -p /content/unmaskd_data/video_raw

!kaggle competitions download deepfake-detection \
    -p /content/unmaskd_data/video_raw \
    --unzip

Kaggle CLI 2.2.4
usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,forums,f,benchmarks,b,config,auth,quota} ...
kaggle: error: unrecognized arguments: --unzip


In [13]:
!rm -rf /content/unmaskd_data/video_raw
!mkdir -p /content/unmaskd_data/video_raw

!kaggle competitions download deepfake-detection \
    -p /content/unmaskd_data/video_raw


100% 9.29G/9.29G [04:39<00:00, 35.7MB/s]



In [15]:
!mkdir -p /content/unmaskd_data/video_extracted

!unzip -q /content/unmaskd_data/video_raw/deepfake-detection.zip \
    -d /content/unmaskd_data/video_extracted

print("Extraction complete!")

Extraction complete!


In [16]:
from pathlib import Path

root = Path('/content/unmaskd_data/video_extracted')

files = [p for p in root.rglob('*') if p.is_file()]

print("Total files:", len(files))
print("\nFirst 30 files:")
for p in files[:30]:
    print(p.relative_to(root))

Total files: 6530

First 30 files:
sample_submission.csv
deep-fake-dataset/test/id21_id28_0001.mp4
deep-fake-dataset/test/id44_0001.mp4
deep-fake-dataset/test/id17_id28_0001.mp4
deep-fake-dataset/test/id30_id23_0007.mp4
deep-fake-dataset/test/id2_0008.mp4
deep-fake-dataset/test/id24_id20_0009.mp4
deep-fake-dataset/test/id29_id30_0009.mp4
deep-fake-dataset/test/00119.mp4
deep-fake-dataset/test/id37_id34_0008.mp4
deep-fake-dataset/test/id43_0002.mp4
deep-fake-dataset/test/00259.mp4
deep-fake-dataset/test/id17_id31_0009.mp4
deep-fake-dataset/test/00095.mp4
deep-fake-dataset/test/id37_id3_0004.mp4
deep-fake-dataset/test/id10_id13_0001.mp4
deep-fake-dataset/test/id6_id2_0008.mp4
deep-fake-dataset/test/id51_id57_0004.mp4
deep-fake-dataset/test/00202.mp4
deep-fake-dataset/test/id51_0005.mp4
deep-fake-dataset/test/id2_id0_0002.mp4
deep-fake-dataset/test/id53_id57_0005.mp4
deep-fake-dataset/test/id53_0000.mp4
deep-fake-dataset/test/id0_0001.mp4
deep-fake-dataset/test/id10_0007.mp4
deep-fake-dat

In [17]:
from pathlib import Path
import pandas as pd

root = Path('/content/unmaskd_data/video_extracted')

print("Top-level contents:")
for p in root.iterdir():
    print(" -", p.name)

print("\nCSV files:")
csv_files = list(root.rglob('*.csv'))

for csv in csv_files:
    print("\nFILE:", csv.relative_to(root))
    try:
        df = pd.read_csv(csv)
        print("Columns:", list(df.columns))
        print("Rows:", len(df))
        print(df.head())
    except Exception as e:
        print("Could not read:", e)

Top-level contents:
 - sample_submission.csv
 - deep-fake-dataset

CSV files:

FILE: sample_submission.csv
Columns: ['id', 'label']
Rows: 518
          id  label
0  00170.mp4      0
1  00208.mp4      0
2  00036.mp4      0
3  00024.mp4      0
4  00063.mp4      0


In [18]:
from pathlib import Path

root = Path('/content/unmaskd_data/video_extracted/deep-fake-dataset')

folders = [
    'Celeb-real',
    'YouTube-real',
    'Celeb-Youtube-fake',
    'test'
]

for folder in folders:
    path = root / folder
    videos = list(path.glob('*.mp4'))
    print(f'{folder}: {len(videos)} videos')

Celeb-real: 482 videos
YouTube-real: 230 videos
Celeb-Youtube-fake: 5299 videos
test: 518 videos


In [19]:
from pathlib import Path
import pandas as pd

root = Path('/content/unmaskd_data/video_extracted/deep-fake-dataset')

rows = []

for folder, label in [
    ('Celeb-real', 1),
    ('YouTube-real', 1),
    ('Celeb-Youtube-fake', 0),
]:
    for video in (root / folder).glob('*.mp4'):
        rows.append({
            'path': str(video),
            'label': label,
            'source': folder,
            'filename': video.name
        })

df = pd.DataFrame(rows)

print("Total training videos:", len(df))
print("\nClass distribution:")
print(df['label'].value_counts().sort_index())

print("\nSource distribution:")
print(df['source'].value_counts())

df.head()

Total training videos: 6011

Class distribution:
label
0    5299
1     712
Name: count, dtype: int64

Source distribution:
source
Celeb-Youtube-fake    5299
Celeb-real             482
YouTube-real           230
Name: count, dtype: int64


,path,label,source,filename
0,/content/unmaskd_data/video_extracted/deep-fak...,1,Celeb-real,332.mp4
1,/content/unmaskd_data/video_extracted/deep-fak...,1,Celeb-real,465.mp4
2,/content/unmaskd_data/video_extracted/deep-fak...,1,Celeb-real,84.mp4
3,/content/unmaskd_data/video_extracted/deep-fak...,1,Celeb-real,41.mp4
4,/content/unmaskd_data/video_extracted/deep-fak...,1,Celeb-real,228.mp4


In [20]:
from pathlib import Path

root = Path('/content/unmaskd_data/video_extracted/deep-fake-dataset')

print("REAL examples:")
for folder in ['Celeb-real', 'YouTube-real']:
    files = list((root / folder).glob('*.mp4'))
    print(f"\n{folder}:")
    for f in files[:10]:
        print(" ", f.name)

print("\nFAKE examples:")
fake_files = list((root / 'Celeb-Youtube-fake').glob('*.mp4'))
for f in fake_files[:30]:
    print(" ", f.name)

REAL examples:

Celeb-real:
  332.mp4
  465.mp4
  84.mp4
  41.mp4
  228.mp4
  386.mp4
  211.mp4
  370.mp4
  35.mp4
  36.mp4

YouTube-real:
  84.mp4
  41.mp4
  228.mp4
  211.mp4
  35.mp4
  36.mp4
  87.mp4
  215.mp4
  104.mp4
  96.mp4

FAKE examples:
  4407.mp4
  3696.mp4
  2524.mp4
  2818.mp4
  3273.mp4
  1563.mp4
  2725.mp4
  1871.mp4
  3335.mp4
  510.mp4
  3823.mp4
  1808.mp4
  3488.mp4
  4258.mp4
  5201.mp4
  5296.mp4
  4976.mp4
  532.mp4
  3446.mp4
  4096.mp4
  4825.mp4
  5159.mp4
  3067.mp4
  4971.mp4
  2020.mp4
  4446.mp4
  2208.mp4
  3824.mp4
  1606.mp4
  3618.mp4


In [21]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

root = Path('/content/unmaskd_data/video_extracted/deep-fake-dataset')

rows = []

# REAL
for folder in ['Celeb-real', 'YouTube-real']:
    for video in (root / folder).glob('*.mp4'):
        rows.append({
            'path': str(video),
            'label': 1,
            'source': folder,
            'filename': video.name
        })

# FAKE
for video in (root / 'Celeb-Youtube-fake').glob('*.mp4'):
    rows.append({
        'path': str(video),
        'label': 0,
        'source': 'Celeb-Youtube-fake',
        'filename': video.name
    })

df = pd.DataFrame(rows)

# Stratified video-level split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df['label']
)

print("TOTAL:", len(df))
print("TRAIN:", len(train_df))
print("VAL:", len(val_df))

print("\nTRAIN:")
print(train_df['label'].value_counts())

print("\nVAL:")
print(val_df['label'].value_counts())

# Save manifests
out = Path('/content/unmaskd_data/manifests')
out.mkdir(parents=True, exist_ok=True)

train_df.to_csv(out / 'train.csv', index=False)
val_df.to_csv(out / 'val.csv', index=False)

print("\nSaved:")
print(out / 'train.csv')
print(out / 'val.csv')

TOTAL: 6011
TRAIN: 5109
VAL: 902

TRAIN:
label
0    4504
1     605
Name: count, dtype: int64

VAL:
label
0    795
1    107
Name: count, dtype: int64

Saved:
/content/unmaskd_data/manifests/train.csv
/content/unmaskd_data/manifests/val.csv


In [6]:
from pathlib import Path

ROOT = Path("/content/unmaskd_data")

print("Checking dataset structure...\n")

# Find all MP4 files
mp4_files = list(ROOT.rglob("*.mp4"))

print("Total MP4 files found:", len(mp4_files))

print("\nFirst 20 MP4 paths:")
for p in mp4_files[:20]:
    print(p)

Checking dataset structure...

Total MP4 files found: 0

First 20 MP4 paths:


In [7]:
from pathlib import Path

print("Files in /content:")
for p in Path("/content").iterdir():
    print(p)

print("\nFiles in /content/unmaskd_data:")
data = Path("/content/unmaskd_data")
if data.exists():
    for p in data.iterdir():
        print(p)
else:
    print("unmaskd_data does not exist")

Files in /content:
/content/.config
/content/unmaskd_data
/content/sample_data

Files in /content/unmaskd_data:
/content/unmaskd_data/manifests
/content/unmaskd_data/frames


In [8]:
from google.colab import files

uploaded = files.upload()


Saving kaggle.json to kaggle.json


In [9]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle credentials installed successfully.")

Kaggle credentials installed successfully.


In [10]:
!mkdir -p /content/unmaskd_data
!kaggle competitions download -c deepfake-detection -p /content/unmaskd_data

100% 9.29G/9.29G [07:47<00:00, 21.3MB/s]



In [11]:
!mkdir -p /content/unmaskd_data
!kaggle competitions download -c deepfake-detection -p /content/unmaskd_data


deepfake-detection.zip: Skipping, found more recently modified local copy (use --force to force download)


In [12]:
import zipfile
from pathlib import Path

zip_path = Path("/content/unmaskd_data/deepfake-detection.zip")
extract_path = Path("/content/unmaskd_data/video_extracted")

extract_path.mkdir(parents=True, exist_ok=True)

print("Extracting dataset...")
print("This may take several minutes.")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction complete!")

Extracting dataset...
This may take several minutes.
Extraction complete!


In [15]:
from pathlib import Path

mp4_files = list(Path("/content/unmaskd_data/video_extracted").rglob("*.mp4"))

print("Total MP4 files found:", len(mp4_files))

for p in mp4_files[:10]:
    print(p)

Total MP4 files found: 6529
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id21_id28_0001.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id44_0001.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id17_id28_0001.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id30_id23_0007.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id2_0008.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id24_id20_0009.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id29_id30_0009.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/00119.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id37_id34_0008.mp4
/content/unmaskd_data/video_extracted/deep-fake-dataset/test/id43_0002.mp4


In [16]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_ROOT = Path("/content/unmaskd_data/video_extracted/deep-fake-dataset")
MANIFEST_DIR = Path("/content/unmaskd_data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

records = []

# REAL videos
for folder in ["Celeb-real", "YouTube-real"]:
    folder_path = DATA_ROOT / folder
    for video in folder_path.glob("*.mp4"):
        records.append({
            "path": str(video),
            "label": 0
        })

# FAKE videos
fake_path = DATA_ROOT / "Celeb-Youtube-fake"
for video in fake_path.glob("*.mp4"):
    records.append({
        "path": str(video),
        "label": 1
    })

df = pd.DataFrame(records)

print("Total labeled videos:", len(df))
print("\nLabels:")
print(df["label"].value_counts())

# 85/15 stratified split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

train_df.to_csv(MANIFEST_DIR / "train.csv", index=False)
val_df.to_csv(MANIFEST_DIR / "val.csv", index=False)

print("\nTRAIN:", len(train_df))
print("VAL:", len(val_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nSaved:")
print(MANIFEST_DIR / "train.csv")
print(MANIFEST_DIR / "val.csv")

Total labeled videos: 6011

Labels:
label
1    5299
0     712
Name: count, dtype: int64

TRAIN: 5109
VAL: 902

Train distribution:
label
1    4504
0     605
Name: count, dtype: int64

Validation distribution:
label
1    795
0    107
Name: count, dtype: int64

Saved:
/content/unmaskd_data/manifests/train.csv
/content/unmaskd_data/manifests/val.csv


In [17]:
import cv2
import pandas as pd
from pathlib import Path
import numpy as np

MANIFEST_DIR = Path("/content/unmaskd_data/manifests")
FRAME_ROOT = Path("/content/unmaskd_data/frames")

FACE_CASCADE = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

FRAMES_PER_VIDEO = 8
IMG_SIZE = 224

def extract_frames(manifest_file, split):
    df = pd.read_csv(manifest_file)

    saved = 0
    failed = 0

    for idx, row in enumerate(df.itertuples(index=False), 1):
        video_path = Path(row.path)
        label = int(row.label)

        label_name = "FAKE" if label == 1 else "REAL"
        output_dir = FRAME_ROOT / split / label_name
        output_dir.mkdir(parents=True, exist_ok=True)

        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            failed += 1
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            cap.release()
            failed += 1
            continue

        positions = np.linspace(
            0,
            total_frames - 1,
            FRAMES_PER_VIDEO,
            dtype=int
        )

        video_saved = 0

        for frame_num in positions:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_num))
            ok, frame = cap.read()

            if not ok:
                continue

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            faces = FACE_CASCADE.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(50, 50)
            )

            if len(faces) > 0:
                # Select largest detected face
                x, y, w, h = max(
                    faces,
                    key=lambda f: f[2] * f[3]
                )

                # Add 20% margin
                margin_x = int(w * 0.20)
                margin_y = int(h * 0.20)

                x1 = max(0, x - margin_x)
                y1 = max(0, y - margin_y)
                x2 = min(frame.shape[1], x + w + margin_x)
                y2 = min(frame.shape[0], y + h + margin_y)

                crop = frame[y1:y2, x1:x2]

            else:
                # Fallback: center crop
                h, w = frame.shape[:2]
                size = min(h, w)

                x1 = (w - size) // 2
                y1 = (h - size) // 2

                crop = frame[
                    y1:y1 + size,
                    x1:x1 + size
                ]

            crop = cv2.resize(
                crop,
                (IMG_SIZE, IMG_SIZE),
                interpolation=cv2.INTER_AREA
            )

            output_file = (
                output_dir /
                f"{video_path.stem}_frame{frame_num}.jpg"
            )

            cv2.imwrite(
                str(output_file),
                crop,
                [cv2.IMWRITE_JPEG_QUALITY, 95]
            )

            saved += 1
            video_saved += 1

        cap.release()

        if video_saved == 0:
            failed += 1

        if idx % 100 == 0:
            print(
                f"{split}: {idx}/{len(df)} videos | "
                f"frames saved: {saved} | failed: {failed}"
            )

    return saved, failed


print("Starting TRAIN extraction...")
train_saved, train_failed = extract_frames(
    MANIFEST_DIR / "train.csv",
    "train"
)

print("\nStarting VALIDATION extraction...")
val_saved, val_failed = extract_frames(
    MANIFEST_DIR / "val.csv",
    "val"
)

print("\n========== EXTRACTION COMPLETE ==========")
print("Training frames:", train_saved)
print("Training failures:", train_failed)
print("Validation frames:", val_saved)
print("Validation failures:", val_failed)

Starting TRAIN extraction...
train: 100/5109 videos | frames saved: 800 | failed: 0
train: 200/5109 videos | frames saved: 1600 | failed: 0
train: 300/5109 videos | frames saved: 2400 | failed: 0
train: 400/5109 videos | frames saved: 3200 | failed: 0
train: 500/5109 videos | frames saved: 4000 | failed: 0
train: 600/5109 videos | frames saved: 4800 | failed: 0
train: 700/5109 videos | frames saved: 5600 | failed: 0
train: 800/5109 videos | frames saved: 6400 | failed: 0
train: 900/5109 videos | frames saved: 7200 | failed: 0
train: 1000/5109 videos | frames saved: 8000 | failed: 0
train: 1100/5109 videos | frames saved: 8800 | failed: 0
train: 1200/5109 videos | frames saved: 9600 | failed: 0
train: 1300/5109 videos | frames saved: 10400 | failed: 0
train: 1400/5109 videos | frames saved: 11200 | failed: 0
train: 1500/5109 videos | frames saved: 12000 | failed: 0
train: 1600/5109 videos | frames saved: 12800 | failed: 0
train: 1700/5109 videos | frames saved: 13600 | failed: 0
train: 

In [1]:
import torch
import torchvision
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path

DATA_ROOT = Path("/content/unmaskd_data/frames")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [4]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_ROOT = Path("/content/unmaskd_data/video_extracted")
MANIFEST_DIR = Path("/content/unmaskd_data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

records = []

# REAL videos
for folder in ["Celeb-real", "YouTube-real"]:
    folder_path = DATA_ROOT / "deep-fake-dataset" / folder
    for video in folder_path.glob("*.mp4"):
        records.append({
            "path": str(video),
            "label": 0
        })

# FAKE videos
fake_path = DATA_ROOT / "deep-fake-dataset" / "Celeb-Youtube-fake"
for video in fake_path.glob("*.mp4"):
    records.append({
        "path": str(video),
        "label": 1
    })

df = pd.DataFrame(records)

print("Total videos:", len(df))
print("\nLabels:")
print(df["label"].value_counts())

# 85/15 stratified split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

train_df.to_csv(MANIFEST_DIR / "train.csv", index=False)
val_df.to_csv(MANIFEST_DIR / "val.csv", index=False)

print("\nTRAIN:", len(train_df))
print("VAL:", len(val_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nFiles created:")
print(MANIFEST_DIR / "train.csv")
print(MANIFEST_DIR / "val.csv")

Total videos: 0

Labels:


KeyError: 'label'

In [3]:
import cv2
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Paths
manifest_dir = Path('/content/unmaskd_data/manifests')
output_dir = Path('/content/unmaskd_data/frames')

train_out = output_dir / 'train'
val_out = output_dir / 'val'

for d in [
    train_out / 'REAL',
    train_out / 'FAKE',
    val_out / 'REAL',
    val_out / 'FAKE'
]:
    d.mkdir(parents=True, exist_ok=True)

# OpenCV face detector
cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def extract_frames(video_path, output_folder, video_id, label,
                   frames_per_video=8):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return 0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return 0

    # Evenly spaced frames
    positions = [
        int(x)
        for x in __import__('numpy').linspace(
            0, total_frames - 1, frames_per_video
        )
    ]

    saved = 0

    for frame_idx, pos in enumerate(positions):

        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        success, frame = cap.read()

        if not success:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        if len(faces) > 0:
            # Select largest detected face
            x, y, w, h = max(
                faces,
                key=lambda box: box[2] * box[3]
            )

            # Add small margin
            margin = int(0.20 * max(w, h))

            x1 = max(0, x - margin)
            y1 = max(0, y - margin)
            x2 = min(frame.shape[1], x + w + margin)
            y2 = min(frame.shape[0], y + h + margin)

            face = frame[y1:y2, x1:x2]

        else:
            # Fallback: use center crop
            h, w = frame.shape[:2]
            size = min(h, w)

            x1 = (w - size) // 2
            y1 = (h - size) // 2

            face = frame[
                y1:y1 + size,
                x1:x1 + size
            ]

        face = cv2.resize(
            face,
            (224, 224),
            interpolation=cv2.INTER_AREA
        )

        filename = (
            f'{video_id}_frame_{frame_idx:02d}.jpg'
        )

        cv2.imwrite(
            str(output_folder / filename),
            face,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

        saved += 1

    cap.release()
    return saved


def process_manifest(csv_path, split):

    df = pd.read_csv(csv_path)

    saved_total = 0

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f'Processing {split}'
    ):

        label_name = 'REAL' if row['label'] == 1 else 'FAKE'

        output_folder = output_dir / split / label_name

        saved_total += extract_frames(
            row['path'],
            output_folder,
            f'{idx:06d}',
            row['label'],
            frames_per_video=8
        )

    return saved_total


train_saved = process_manifest(
    manifest_dir / 'train.csv',
    'train'
)

val_saved = process_manifest(
    manifest_dir / 'val.csv',
    'val'
)

print("\nExtraction complete!")
print("Training frames:", train_saved)
print("Validation frames:", val_saved)

FileNotFoundError: [Errno 2] No such file or directory: '/content/unmaskd_data/manifests/train.csv'

## 4. Prepare a small verified training subset

The existing `prepare_ffpp.py` expects FaceForensics++-style folders. Because the Kaggle competition layout can differ, inspect the output above first. If the dataset contains an explicit metadata/labels file, adapt the preparation step to that metadata. Do not infer labels from arbitrary filenames.


In [ ]:
# Find likely metadata files. Review the printed candidates before using one.
for p in files:
    if p.suffix.lower() in {'.csv','.json','.txt'}:
        print(p.relative_to(root))


### If your downloaded dataset matches the expected REAL/FAKE directory layout

Run the following. Otherwise stop here and adapt the preparation script to the inspected layout.


In [ ]:
# Example only — execute after confirming the folder layout.
# !python training/prepare_ffpp.py \
#   --dataset-root /content/unmaskd_data/video_raw \
#   --output-dir /content/unmaskd_data/video_frames \
#   --frames-per-video 8


## 5. Train the first video baseline

Once `/content/unmaskd_data/video_frames` is an ImageFolder dataset with `train/val/test` and `real/fake` classes, run the baseline. The script reports real metrics from the held-out test set and saves a checkpoint.


In [ ]:
from pathlib import Path
data_dir = Path('/content/unmaskd_data/video_frames')
if data_dir.exists():
    print('Prepared dataset found:', data_dir)
    print('Subdirectories:', [p.name for p in data_dir.iterdir() if p.is_dir()])
else:
    print('Prepared dataset not found yet. Complete the verified preparation step first.')


In [ ]:
# Start with a small run to validate the pipeline. Increase epochs/batch size after it works.
# !python training/train_video.py \
#   --data-dir /content/unmaskd_data/video_frames \
#   --epochs 5 \
#   --batch-size 32


## 6. Audio dataset — ASVspoof 2021 DF

ASVspoof 2021's Speech Deepfake (DF) task contains bona-fide and spoofed speech. Registration and dataset access are handled by ASVspoof/Zenodo. Download it separately and keep it outside GitHub.

After download, create a manifest with: `path`, `label` (`bonafide`/`spoof`), `speaker/source_id`, and `split`. Use speaker/source-disjoint splits.


In [ ]:
# Placeholder for the ASVspoof DF path after you download and extract it.
AUDIO_ROOT = Path('/content/unmaskd_data/asvspoof_df')
print('Expected audio dataset location:', AUDIO_ROOT)
print('Next stage: build the audio manifest and train the anti-spoof model.')


## 7. Save training artifacts

Keep checkpoints and metrics in `/content/unmaskd_artifacts` during training. Upload only selected artifacts/configs to GitHub or approved storage. Never upload raw datasets or `kaggle.json`.


In [ ]:
!mkdir -p /content/unmaskd_artifacts
print('Artifact directory ready:', '/content/unmaskd_artifacts')


## Target Unmaskd live-call pipeline

Video detector + audio anti-spoof detector → calibrated probabilities → signal-quality checks → temporal smoothing → validated multimodal fusion → authenticity score.

The live UI currently uses short rolling media windows. A production model should be trained with compression, frame drops, blur, resizing, lighting variation, microphone noise, echo/reverb, resampling and other WebRTC/VoIP-like conditions.
